# Estimability and Profile Likelihood: Historical Data vs Historical + Lot 1

This notebook compares parameter estimability before and after adding the executed Lot 1 data.

The workflow follows the same model-based design logic used in the campaign:

1. Load historical normalized natural/synthetic must data.
2. Reconstruct Lot 1 as executable model batches using actual initial states, sampling times and pulse logs.
3. Fit the core kinetic parameter block for both information sets.
4. Compute FIM, covariance approximations, eigenvalue diagnostics and weak directions.
5. Run profile likelihood for selected weak core kinetic parameters.

The extended FIM includes observed core and secondary states. Online CO2 is included as an information-time block, not as a final flow-calibrated likelihood, because the current model state is cumulative CO2 while the sensor reports gas flow.

The executed notebook uses a screening profile likelihood for `Kd0` and `qN` to keep the old-vs-new comparison operational. To make a more exhaustive profile, rerun the analysis cell with more parameters, for example `("Kd0", "qN", "qEG", "betaG0")`, and increase `grid_points` and `profile_nfev`.

The core fit uses a small deterministic polish/multistart around `qN` and `Kd0`. This is included because the profile likelihood can otherwise reveal a lower objective than the initial least-squares solution.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

import run_estimability_old_vs_lot1 as analysis

RESULTS = Path("results") / "estimability_old_vs_lot1"


## Run Analysis

In [ ]:
analysis.run_analysis(
    profile_parameters=("Kd0", "qN"),
    grid_points=3,
    fit_nfev=300,
    profile_nfev=45,
    step=0.04,
)

## Batch Inventory

In [ ]:
batch_summary = pd.read_csv(RESULTS / "batch_summary.csv")
display(batch_summary)

## FIM Metrics

In [ ]:
fit_candidates = pd.read_csv(RESULTS / "core_fit_candidate_summary.csv")
display(fit_candidates[["case", "seed", "final_wsse", "nfev", "success"]])

fim_metrics = pd.read_csv(RESULTS / "fim_metrics_by_case.csv")
display(fim_metrics)

## Extended Estimability Comparison

In [ ]:
estimability = pd.read_csv(RESULTS / "estimability_extended_by_case.csv")
change = pd.read_csv(RESULTS / "estimability_change_old_vs_plus_lot1.csv")
display(estimability.sort_values(["case", "std_log_approx"]))
display(change.sort_values("std_log_delta_plus_minus_old"))

In [ ]:
display(Image(filename=str(RESULTS / "figures" / "estimability_std_log_comparison.png")))

## Weak Eigen Directions

In [ ]:
weak = pd.read_csv(RESULTS / "weak_directions_extended_by_case.csv")
display(weak)

## Profile Likelihood

In [ ]:
profile_summary = pd.read_csv(RESULTS / "profile_likelihood_summary_core_by_case.csv")
profile = pd.read_csv(RESULTS / "profile_likelihood_core_by_case.csv")
display(profile_summary)
display(profile.head(30))

In [ ]:
display(Image(filename=str(RESULTS / "figures" / "profile_likelihood_historical_only.png")))
display(Image(filename=str(RESULTS / "figures" / "profile_likelihood_historical_plus_lot1.png")))

## Report

In [ ]:
report = (RESULTS / "estimability_old_vs_lot1_report.md").read_text(encoding="utf-8")
print(report)